In [ ]:
# processing for a single batch (run for each of the batches)

import pandas as pd
import matplotlib.pylab as plt
import os
import importlib
import analysis_utils
import numpy as np

data_dir = "../human-data/play-exp/human-v-human/raw-data/"
subdir = "final-batch10"

pth = f"{data_dir}{subdir}/"
data_files = {f.split(".csv")[0]: pd.read_csv(f"{pth}{f}") for f in os.listdir(f"{pth}")}


In [ ]:
uid2prolific = {}
uid2comments = {}
arena2players = {}
arena2judgement = {}

for _, entry in data_files["player"].iterrows():
    dt = entry["endedLastChangedAt"]
    entered_id = entry.participantIdentifier
    uid = entry.id
    arena_id = entry.gameID
    comments = entry.exitSurvey

    if str(comments) == "nan" or len(eval(comments)) == 0:
        continue

    if len(entered_id) != 24 or str(arena_id) == "nan":
        continue
    uid2prolific[uid] = entered_id
    uid2comments[uid] = eval(comments)

    if arena_id not in arena2players:
        arena2players[arena_id] = {uid}
    else:
        arena2players[arena_id].add(uid)

    if "advantage" in entry["treatmentName"]:
        arena2judgement[arena_id] = "exp_utility"
    else:
        arena2judgement[arena_id] = "fun"

uids = sorted(list(uid2prolific.keys()))
player2arena = {uid: arena_id for arena_id, players in arena2players.items() for uid in players}


In [ ]:
round_df = data_files["round"]
arena2stimuli = {}
arena2game_idx = {}

for arena_id in arena2players:
    arena_df = round_df.loc[round_df.gameID == arena_id]

    stimuli = list(arena_df.stimuliID)

    sub_uids = arena2players[arena_id]
    if len(sub_uids) <= 1:
        for uid in sub_uids:
            uids.remove(uid)
        continue
    arena2stimuli[arena_id] = stimuli
    arena2game_idx[arena_id] = {stim: game_idx for game_idx, stim in enumerate(stimuli)}


In [ ]:
uid2judgements = {uid: {} for uid in uids}
player_orderings = {uid: {} for uid in uids}
slider_df = data_files["playerRound"]

arena2players = {arena: set() for arena in arena2game_idx}

for uid in uids:
    games_played = arena2stimuli[player2arena[uid]]
    judge_df = slider_df.loc[slider_df.playerID == uid].reset_index()
    for game_idx, entry in judge_df.iterrows():
        if entry["gameOutcome"] == "TBD":
            continue
        val = []

        stim_id = entry["stimuliID"]

        if "judgmentDraw" in entry and str(entry["judgmentDraw"]) != "nan":
            val_draw = entry["judgmentDraw"]
            val_firstplayer = entry["judgmentAdvantage"]
            val = [val_firstplayer, val_draw]
        else:
            if "judgment" in entry:
                val = [entry["judgment"]]
            else:
                val = ["nan", "nan"]

        if "judgmentSkill" in entry:
            val_skill = entry["judgmentSkill"]
        else:
            val_skill = -1

        val.append(val_skill)

        outcome = entry["gameOutcome"]
        order = entry["playerOrder"]

        player_orderings[uid][stim_id] = order
        arena = player2arena[uid]
        arena2players[arena].add(uid)
        stim_df = round_df.loc[round_df.stimuliID == stim_id]
        arena_df = stim_df.loc[stim_df.gameID == player2arena[uid]]
        player_orders = eval(arena_df["playerOrders"].iloc[0])
        if player_orders[uid] != order:
            print(stim_id, order, player_orders)

        uid2judgements[uid][stim_id] = {
            "val": val,
            "order": order,
            "outcome": outcome,
            "game_order": game_idx,
        }


In [ ]:
import bisect


def find_closest_index(sorted_list, target):
    idx = bisect.bisect_left(sorted_list, target)
    if idx == 0:
        return 0
    if idx == len(sorted_list):
        return len(sorted_list) - 1
    if target < sorted_list[idx]:
        return idx - 1
    else:
        return idx


In [ ]:
importlib.reload(analysis_utils)

all_stimuli = list(set().union(*arena2stimuli.values()))

all_stimuli_data = {stim: {
    "arena": [],
    "start": [],
    "duration": [],
    "move_times": [],
    "boards": [],
    "draw_requests": [],
    "draw_rejects": [],
    "draw_accepts": [],
    "draw_info": [],
    "outcome": [],
    "judgements": [],
    "player_judgements": [],
    "player_exp_payoffs": [],
    "order2player": [],
    "judgement_type": [],
    "surrender_info": [],
} for stim in all_stimuli}

main_fig_dir = "figs"

win_judgements = []
loss_judgements = []
draw_judgements = []
save_arena_figs = True

stim2draw_accepts = {stim: 0 for stim in all_stimuli}


def switch_player(player, players):
    return [p for p in players if player != p][0]


for i, stim in enumerate(all_stimuli):
    stim_df = round_df.loc[round_df.stimuliID == stim]
    for j, entry in stim_df.iterrows():
        arena = entry.gameID
        if arena not in arena2players:
            continue

        judgement_type = arena2judgement[arena]

        start = entry.startLastChangedAt
        end = entry.endedLastChangedAt
        round_start = analysis_utils.parse_time(start)
        round_end = analysis_utils.parse_time(end)
        duration = round_end - round_start
        all_boards = entry.allBoards
        outcome = entry.gameWinner

        if str(entry.moveTimes) == "nan":
            print("ERROR")
            continue
        raw_move_times = eval(entry.moveTimes)
        if len(raw_move_times) == 0:
            print(arena, i, j)
        move_times = []
        move_start_time = 0
        for move_end_time in raw_move_times:
            move_time_seconds = (move_end_time - move_start_time) / 1000
            move_times.append(move_time_seconds)
            move_start_time = move_end_time

        surrenders_made = entry.surrenderMade
        if str(surrenders_made) != "nan":
            surrenders_made = eval(entry.surrenderMade)
            surrender_player, surrender_time = surrenders_made
            idx = find_closest_index(raw_move_times, surrender_time)
            if len(eval(all_boards)) != 0:
                surrender_board = eval(all_boards)[idx]
            else:
                surrender_board = None

            if "7.0*7.0" in stim and arena == "01JKXV9AKP0F550KZB23PTMDY6":
                surrender_board = eval(all_boards)[-2]

            surrender_info = [surrender_player, surrender_time, surrender_board]
        else:
            surrender_info = None

        draws = eval(entry.allDrawAccepts)
        draw_requests = eval(entry.allDrawRequests)
        rejects = eval(entry.allDrawRejects)

        draw_info = {"reject": [], "accept": []}
        if len(draws) != 0:
            stim2draw_accepts[stim] += 1

            player_request, request_time = draw_requests[-1]
            idx = find_closest_index(raw_move_times, draw_requests[-1][1])
            if len(eval(all_boards)) == 0:
                request_board = [[]]
            else:
                request_board = eval(all_boards)[idx]
                draw_info["accept"].append([player_request, request_time, request_board])

        if len(draws) == 0 and len(draw_requests) > 0:

            req_entries = [req_time for _, req_time in draw_requests]
            for reject_time in rejects:
                req_idx = find_closest_index(req_entries, reject_time)
                req_entry = draw_requests[req_idx]
                player_request, request_time = req_entry
                idx = find_closest_index(raw_move_times, request_time)
                request_board = eval(all_boards)[idx]
                draw_info["reject"].append([player_request, request_time, request_board])

            leftover_req = [
                req_entry
                for req_entry in draw_requests
                if len(rejects) == 0
                or np.any([
                    move_time > req_entry[1] and req_entry[1] > np.max(rejects)
                    for move_time in raw_move_times
                ])
            ]

            if len(leftover_req) != 0:
                for player_request, request_time in leftover_req:
                    idx = find_closest_index(raw_move_times, request_time)
                    request_board = eval(all_boards)[idx]
                    draw_info["reject"].append([player_request, request_time, request_board])

        n_rows, n_cols, win_conds = stim.split("*")
        n_rows = int(float(n_rows))
        n_cols = int(float(n_cols))
        if arena not in arena2game_idx:
            continue
        game_order = arena2game_idx[arena][stim]

        players = sorted(arena2players[arena])
        if len(players) == 0:
            continue

        colors = ["purple", "green"]
        player_colors = {player: colors[player_idx] for player_idx, player in enumerate(players)}
        order2player = {player_orderings[player][stim]: player for player in players}
        player2order = {v: k for k, v in order2player.items()}

        player_judgements = {player: uid2judgements[player][stim]["val"] for player in players}

        if outcome == 0.0:
            outcome = "Draw"
            vals = list(player_judgements.values())
            draw_judgements.extend(vals)
        else:
            outcome = order2player[outcome]
            win_judgements.append(player_judgements[outcome])
            loss_judgements.append(player_judgements[switch_player(outcome, players)])

        player_exp_payoffs = {}

        if judgement_type == "exp_utility":

            for player, (pfirst, pdraw, pskill) in player_judgements.items():
                if str(pdraw) == "nan" or str(pskill) == "nan":
                    continue
                pwin = analysis_utils.get_pwin(pdraw, pfirst)
                pdraw /= 100
                pwin /= 100
                payoff = analysis_utils.compute_utility(pdraw, pwin)
                player_exp_payoffs[player] = payoff

        try:
            player_move_map, move_order_map = analysis_utils.process_game_moves(all_boards)
            fig = analysis_utils.construct_board(
                player_move_map, move_order_map, n_rows, n_cols, win_conds,
                draw_accepted=len(draws) > 0,
                player_colors=player_colors, player_orders=order2player,
                player_judgements=player_judgements,
                game_outcome=outcome,
            )
        except:
            print("error: ", arena)

        all_stimuli_data[stim]["arena"].append(arena)
        all_stimuli_data[stim]["boards"].append(all_boards)
        all_stimuli_data[stim]["outcome"].append(outcome)
        all_stimuli_data[stim]["duration"].append(duration)
        all_stimuli_data[stim]["move_times"].append(move_times)
        all_stimuli_data[stim]["judgements"].append(list(player_judgements.values()))
        all_stimuli_data[stim]["player_judgements"].append(player_judgements)
        all_stimuli_data[stim]["player_exp_payoffs"].append(player_exp_payoffs)
        all_stimuli_data[stim]["order2player"].append(order2player)
        all_stimuli_data[stim]["draw_requests"].append(draw_requests)
        all_stimuli_data[stim]["judgement_type"].append(judgement_type)
        all_stimuli_data[stim]["draw_accepts"].append(draws)
        all_stimuli_data[stim]["draw_info"].append(draw_info)
        all_stimuli_data[stim]["surrender_info"].append(surrender_info)

        if save_arena_figs:
            fig_dir = f"{main_fig_dir}/{arena}/"
            if not os.path.exists(fig_dir):
                os.makedirs(fig_dir)
            fig.tight_layout()
            fig.savefig(f"{fig_dir}/game_idx_{game_order}_{stim}.png", dpi=400)
        plt.close()


In [ ]:
import json

with open(
    f"/Users/katiecollins/intuitive-game-reasoning/human-data/play-exp/human-v-human/final-play/humanhuman_{subdir}.json",
    "w",
) as f:
    json.dump(all_stimuli_data, f)


In [ ]:
import json

with open("/Users/katiecollins/intuitive-game-reasoning/human-data/think-exp/exp_utility_cogsci_res.json", "r") as f:
    orig_res = json.load(f)

with open("../model-data/think-exp/processed_payoff_subset.json", "r") as f:
    model_res_payoffs = json.load(f)


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 8))
axes = axes.flatten()

subcond = "The second player can place 2 pieces as their first move, while the first player can only place 1 piece as their first move."
subcond2 = "The first player can place 2 pieces as their first move, while the second player can only place 1 piece as their first move."

new_payoffs = []
orig_payoffs = []
m_payoffs = []
payoffs_stims = []

for i, (stim, stim_data) in enumerate(all_stimuli_data.items()):
    orig_stim = stim.split(" Horizontal, vertical, and diagonal all count.")[0]
    payoffs_stims.append(stim)

    n_rows, n_cols, win_conds = stim.split("*")
    n_rows = int(float(n_rows))
    n_cols = int(float(n_cols))
    ax = axes[i]
    p1_payoffs = []
    p1_skills = []
    p2_payoffs = []
    p2_skills = []
    outcomes = []
    for payoff_data, skill_data, order2player, outcome in zip(
        stim_data["player_exp_payoffs"], stim_data["player_judgements"],
        stim_data["order2player"], stim_data["outcome"],
    ):
        player2order = {v: k for k, v in order2player.items()}
        for player in payoff_data:
            payoff = payoff_data[player]
            _, _, skill = skill_data[player]
            order = player2order[player]

            if str(skill) == "nan":
                print("missing: ", player, stim)
                skill = 50
            if order == 1:
                p1_payoffs.append(payoff)
                p1_skills.append(skill)
            else:
                p2_payoffs.append(payoff)
                p2_skills.append(skill)

        if outcome == "Draw":
            outcomes.append(0)
        else:
            outcomes.append(player2order[outcome])

    orig_agg = np.mean(orig_res[orig_stim]["payoff"])
    agg_vals = p1_payoffs + p2_payoffs
    new_mean = np.mean(agg_vals)

    new_payoffs.append(agg_vals)
    orig_payoffs.append(orig_res[orig_stim]["payoff"])
    m_payoffs.append(model_res_payoffs["ours"][orig_stim])

    for x, y, o in zip(p1_skills, p1_payoffs, outcomes):
        if o == 0: shape = "o"
        elif o == 1: shape = "x"
        else: shape = "^"
        ax.scatter(x, y, marker=shape, alpha=0.5, s=120, c="black")
    for x, y, o in zip(p2_skills, p2_payoffs, outcomes):
        if o == 0: shape = "o"
        elif o == 1: shape = "x"
        else: shape = "^"
        ax.scatter(x, y, marker=shape, alpha=0.5, s=120, c="black")

    linewidth = 2.5
    alpha = 0.7
    ax.axhline(orig_agg, color="blue", linewidth=linewidth, alpha=alpha)
    ax.axhline(new_mean, color="green", linewidth=linewidth, alpha=alpha)
    ax.axhline(np.mean(model_res_payoffs["expert_mcts"][orig_stim]), color="grey", linewidth=linewidth, alpha=alpha)
    ax.axhline(np.mean(model_res_payoffs["ours"][orig_stim]), color="purple", linewidth=linewidth, alpha=alpha)

    ax.set_ylim([-1.1, 1.1])
    ax.set_xlim([0, 100])
    ax.set_title(f"{n_rows}x{n_cols}\n{win_conds.replace(subcond, chr(10) + 'Second player plays twice.').replace(subcond2, chr(10) + 'First player plays twice.')}")
    ax.set_xlabel("Skill", fontsize=18)
    ax.set_ylabel("Payoff", fontsize=18)
plt.subplots_adjust(hspace=0.7, wspace=0.4)


In [ ]:
import seaborn as sns
from collections import defaultdict

fig, axes = plt.subplots(3, 3, figsize=(18, 8))
axes = axes.flatten()

subcond = "The second player can place 2 pieces as their first move, while the first player can only place 1 piece as their first move."
subcond2 = "The first player can place 2 pieces as their first move, while the second player can only place 1 piece as their first move."

stim2outcomes = {}

for i, (stim, stim_data) in enumerate(all_stimuli_data.items()):
    stim = stim.split(" Horizontal, vertical, and diagonal all count.")[0]
    n_rows, n_cols, win_conds = stim.split("*")
    n_rows = int(float(n_rows))
    n_cols = int(float(n_cols))
    ax = axes[i]

    outcomes_payoffs = defaultdict(list)

    for payoff_data, outcome, order2player in zip(
        stim_data["player_exp_payoffs"],
        stim_data["outcome"],
        stim_data["order2player"],
    ):
        player2order = {v: k for k, v in order2player.items()}

        for player in payoff_data:
            payoff = payoff_data[player]
            order = player2order[player]

            if outcome == "Draw":
                outcome_category = "Draw"
            elif player == outcome:
                outcome_category = "P1" if order == 1 else "P2"
            else:
                outcome_category = "P2" if order == 1 else "P1"

            outcomes_payoffs[outcome_category].append(payoff)

    outcomes = ["P1", "P2", "Draw"]
    x_positions = np.arange(len(outcomes))
    stim2outcomes[stim] = outcomes_payoffs
    width = 0.4

    means = [np.mean(outcomes_payoffs[o]) if outcomes_payoffs[o] else np.nan for o in outcomes]

    ax.bar(x_positions, means, width, color="lightblue", alpha=0.7)

    linewidth = 2.5
    alpha = 0.7
    orig_agg = np.mean(orig_res[stim]["payoff"])
    new_mean = np.mean([val for outcome_list in outcomes_payoffs.values() for val in outcome_list])

    ax.axhline(orig_agg, color="blue", linewidth=linewidth, alpha=alpha, linestyle="--", label="Original Mean")
    ax.axhline(new_mean, color="green", linewidth=linewidth, alpha=alpha, linestyle="--", label="New Mean")
    ax.axhline(np.mean(model_res_payoffs["expert_mcts"][stim]), color="grey",
               linewidth=linewidth, alpha=alpha, linestyle="--", label="Expert MCTS")
    ax.axhline(np.mean(model_res_payoffs["ours"][stim]), color="purple",
               linewidth=linewidth, alpha=alpha, linestyle="--", label="Our Model")

    ax.set_ylim([-1.1, 1.1])
    ax.set_xticks(x_positions)
    ax.set_xticklabels(outcomes)
    ax.set_title(f"{n_rows}x{n_cols}\n{win_conds.replace(subcond, chr(10) + 'Second player plays twice.').replace(subcond2, chr(10) + 'First player plays twice.')}")
    ax.set_xlabel("Winner", fontsize=12)
    ax.set_ylabel("Payoff", fontsize=12)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.subplots_adjust(hspace=0.7, wspace=0.4)
plt.tight_layout()
plt.show()


In [ ]:
filtered_new_payoffs = []
for i, payoffs in enumerate(new_payoffs):
    mean_val = np.mean(payoffs)
    distances = np.abs(payoffs - mean_val)
    sorted_indices = np.argsort(distances)
    payoffs = np.array(payoffs)[sorted_indices[:-2]]
    filtered_new_payoffs.append(payoffs)
    if np.mean(payoffs) < -0.1:
        print(payoffs_stims[i], "\n\t", np.mean(payoffs), np.mean(orig_payoffs[i]), np.mean(m_payoffs[i]))

p1 = [np.mean(p) for p in orig_payoffs]
p2 = [np.mean(p) for p in filtered_new_payoffs]
p3 = [np.mean(p) for p in m_payoffs]
plt.scatter(p2, p1, color="blue", alpha=0.5, s=100)
plt.errorbar(p2, p1,
             xerr=[analysis_utils.compute_se(p) for p in filtered_new_payoffs],
             yerr=[analysis_utils.compute_se(p) for p in orig_payoffs],
             fmt="o", color="blue", alpha=0.5)
plt.scatter(p2, p3, color="red", alpha=0.5, s=100)
plt.errorbar(p2, p3,
             xerr=[analysis_utils.compute_se(p) for p in filtered_new_payoffs],
             yerr=[analysis_utils.compute_se(p) for p in m_payoffs],
             fmt="o", color="red", alpha=0.5)
plt.plot([-1.05, 1.05], [-1.05, 1.05], "k--", alpha=0.5)
plt.ylabel("Just Think Payoffs", fontsize=18)
plt.xlabel("Play Payoffs", fontsize=18)
plt.xlim([-1.05, 1.05])
plt.ylim([-1.05, 1.05])


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 6))

axes = axes.flatten()
for i, (stim, stim_data) in enumerate(all_stimuli_data.items()):
    ax = axes[i]
    player_judgements = stim_data["player_exp_payoffs"]
    orig_judgements = orig_res[stim]["payoff"]
    ax.hist(orig_judgements, label="just think", density=True, alpha=0.8)
    player_vals = []
    for entry in player_judgements:
        player_vals.extend([v for v in entry.values() if str(v) != "nan"])
    ax.hist(player_vals, label="play", density=True, alpha=0.8)
    ax.set_title(stim)
    ax.legend()
    ax.set_ylabel("Density", fontsize=14)
    if i >= 4:
        ax.set_xlabel("Payoff", fontsize=14)
    ax.set_xlim([-1.05, 1.05])

plt.subplots_adjust(hspace=0.5, wspace=0.3)


In [ ]:
importlib.reload(analysis_utils)

for stim, stimuli_data in all_stimuli_data.items():
    all_move_times = stimuli_data["move_times"]
    max_num_moves = np.max([len(move_times) for move_times in all_move_times])
    move_idxs = np.arange(max_num_moves)

    move_stats = [[] for _ in range(max_num_moves)]

    for _, move_times in enumerate(all_move_times):
        for i, move_time in enumerate(move_times):
            move_stats[i].append(move_time)

    plt.figure()
    plt.title(f"{stim}")
    means = [np.mean(t) for t in move_stats]
    se = analysis_utils.compute_se_bars(move_stats)
    plt.scatter(move_idxs, means, s=100)
    plt.errorbar(move_idxs, means, yerr=se, capsize=0, ls="none", color="black", elinewidth=2)
    plt.xlabel("Move Number", fontsize=18)
    plt.ylabel("Time (seconds)", fontsize=18)


In [ ]:
import json

with open("orig_cogsci_fun.json", "r") as f:
    orig_cogsci_fun = json.load(f)


In [ ]:
importlib.reload(analysis_utils)

max_n = len(player2arena)

for stim, stimuli_data in all_stimuli_data.items():
    all_judgements = analysis_utils.flatten(stimuli_data["judgements"])

    plt.figure()
    alpha = 0.5
    bins = 10

    pre_judgements = orig_cogsci_fun[stim]
    plt.hist(pre_judgements, bins=bins, alpha=alpha, label=f"Pre [Orig] (N={len(pre_judgements)})")

    plt.hist(all_judgements, bins=bins, alpha=alpha, label=f"Post-Play (N={len(all_judgements)})")
    plt.legend()

    plt.xlim([-5, 105])
    plt.ylim([0, max_n])
    plt.title(f"{stim}")

    plt.xlabel("Funness", fontsize=18)
    plt.ylabel("Count", fontsize=18)


In [ ]:
all_game_lens = []
all_funness = []

len_type = "num_moves"

draw_fun_judgements = []
win_fun_judgements = []
loss_fun_judgements = []

win_loss_lens = []
draw_lens = []

for stim, stimuli_data in all_stimuli_data.items():

    for (outcome, order2player,
         player_judgements, boards, move_times) in zip(
        stimuli_data["outcome"],
        stimuli_data["order2player"],
        stimuli_data["player_judgements"],
        stimuli_data["boards"],
        stimuli_data["move_times"],
    ):

        if len_type == "num_moves":
            game_len = len(eval(boards))
        else:
            game_len = np.sum(move_times)

        all_game_lens.extend([game_len, game_len])

        if outcome == "Draw":
            fun_vals = list(player_judgements.values())
            draw_fun_judgements.extend(fun_vals)
            draw_lens.extend([game_len, game_len])
            all_funness.extend(fun_vals)
        else:
            winner_fun = player_judgements[outcome]
            loser_fun = player_judgements[switch_player(outcome, player_judgements.keys())]
            win_fun_judgements.append(winner_fun)
            loss_fun_judgements.append(loser_fun)
            win_loss_lens.append(game_len)
            all_funness.extend([winner_fun, loser_fun])

size = 100
alpha = 0.7

plt.scatter(win_loss_lens, win_fun_judgements, alpha=alpha, label="Won", color="purple", s=size)
plt.scatter(win_loss_lens, loss_fun_judgements, alpha=alpha, label="Lost", color="green", s=size)
plt.scatter(draw_lens, draw_fun_judgements, alpha=alpha, label="Draw", color="orange", s=size)

plt.ylim([-5, 105])
plt.legend()
plt.ylabel("Funness", fontsize=18)
plt.xlabel("Game Length", fontsize=18)
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2),
           ncol=3, fancybox=True, shadow=True, fontsize=18)
plt.tight_layout()
plt.savefig("funness_w_outcomes_gamelen.png", dpi=400)
